# 05 — CTU-UHB External Validation — Feasibility Test

Initial feasibility test for CTU-UHB external validation. Extracts approximate 
ASTV/AC analogues from raw WFDB signal data to assess whether SisPorto-style 
features can be derived from raw FHR recordings. 

See Preliminary Report Section 4 for discussion of findings.

**Data**: not included in this repo — see `data/external/README.md` for download instructions.

In [ ]:
import os
import wfdb
import numpy as np

# Update this path to wherever you've downloaded CTU-UHB records locally
# See data/external/README.md for download instructions
folder = 'data/external/ctu-uhb-ctgdb-1.0.0'

# list available records
hea_files = [f for f in os.listdir(folder) if f.endswith('.hea')]
print("Found records:", sorted(hea_files)[:10])

# load the first one to inspect structure
record_name = sorted(hea_files)[0].replace('.hea', '')
record_path = os.path.join(folder, record_name)
record = wfdb.rdrecord(record_path)

print("Signal names:", record.sig_name)
print("Sampling frequency:", record.fs)
print("Signal shape:", record.p_signal.shape)
print("First 10 values of channel 0 (FHR):", record.p_signal[:10, 0])

Found records: ['1001.hea', '1002.hea', '1003.hea', '1004.hea', '1005.hea', '1006.hea', '1007.hea', '1008.hea', '1009.hea', '1010.hea']
Signal names: ['FHR', 'UC']
Sampling frequency: 4
Signal shape: (19200, 2)
First 10 values of channel 0 (FHR): [150.5  150.5  151.   151.25 151.25 150.25 150.25 150.25 148.75 148.75]


In [4]:
def compute_astv_analogue(fhr, fs, window_sec=60):
    """Approximate Short-Term Variability: mean absolute difference 
    between consecutive samples, windowed."""
    window_size = int(window_sec * fs)
    fhr_clean = fhr[fhr > 50]  # remove dropout/zero artifacts
    diffs = np.abs(np.diff(fhr_clean))
    stv_windows = [diffs[i:i+window_size].mean() 
                   for i in range(0, len(diffs), window_size) if len(diffs[i:i+window_size]) > 0]
    return np.mean(stv_windows) if stv_windows else None


def compute_ac_analogue(fhr, fs, baseline_window=600):
    """Approximate Accelerations: count of rises >=15bpm above 
    local baseline lasting >=15 seconds."""
    baseline = np.median(fhr[fhr > 50])
    threshold = baseline + 15
    above = fhr > threshold
    min_samples = int(15 * fs)
    count, run_len = 0, 0
    for val in above:
        if val:
            run_len += 1
        else:
            if run_len >= min_samples:
                count += 1
            run_len = 0
    return count

In [5]:
# Run feasibility test on a sample of 3 records
record_ids = ['1001', '1002', '1003']

for rid in record_ids:
    record_path = os.path.join(folder, rid)
    rec = wfdb.rdrecord(record_path)
    fhr = rec.p_signal[:, 0]  # FHR channel
    fs = rec.fs
    
    astv = compute_astv_analogue(fhr, fs)
    ac = compute_ac_analogue(fhr, fs)
    
    print(f"Record {rid}: ASTV-analogue={astv:.2f}, AC-analogue={ac}")

Record 1001: ASTV-analogue=0.65, AC-analogue=10
Record 1002: ASTV-analogue=0.69, AC-analogue=8
Record 1003: ASTV-analogue=0.64, AC-analogue=5


## Finding

Signal-derived analogues can be computed directly from raw FHR data, but values 
are not on the same scale as UCI's SisPorto-derived ASTV (which reports variability 
as a percentage-of-time metric, not mean absolute bpm difference). Reconciling 
this requires replicating SisPorto's specific calculation methodology — identified 
as the primary remaining task for the Interim Report phase.